### Algoritmo Propuesta

In [14]:
# librerias

import copy
from collections import defaultdict
import random
import numpy as np
import json

In [15]:
# Clase: Distribuidor de estudiantes a desafíos

class GenerarEquipos:
    def __init__(self, estudiantes, desafios, carreras):
        self.estudiantes = estudiantes
        self.desafios = desafios
        self.carreras = {carrera['Nombre']: carrera['Maximo'] for carrera in carreras}
        self.asignaciones = defaultdict(list)
        self.asignacion_estudiantes = {}

    # Verificar restricciones de carrera y tamaño del equipo antes de asignar al estudiante
    def verificar_asignacion_valida(self, desafio, estudiante):
        equipo_actual= self.asignaciones[desafio]

        # Verificar tamaño máximo del equipo
        if len(equipo_actual) >= 4:
            return False
        
        # Contar estudiantes por carrera en el equipo actual
        cuantos_por_carrera = defaultdict(int)
        for miembro in equipo_actual:
            cuantos_por_carrera[miembro['Carrera']] += 1

        # Verificar restricciones de carrera según la configuración
        carrera_del_estudiante = estudiante['Carrera']
        max_permitido = self.carreras.get(carrera_del_estudiante, 1)
        if cuantos_por_carrera[carrera_del_estudiante] >= max_permitido:
            return False
        
        return True
    
    # Aleatorizar orden de estudiantes
    def estudiantes_lista_random(self):        
        random.shuffle(self.estudiantes)

    # Primera fase: asignar por primera preferencia
    def asignacion_inicial(self):
        estudiantes_copia = self.estudiantes.copy()

        for estudiante in estudiantes_copia:
            if len(estudiante['Postulaciones']) > 0:
                preferencias = estudiante['Postulaciones'].copy()
                
                for preferencia in preferencias:
                    if self.verificar_asignacion_valida(preferencia, estudiante):
                        self.asignaciones[preferencia].append(estudiante)
                        self.asignacion_estudiantes[estudiante['Nombre']] = preferencia
                        break

    def encontrar_mejor_equipo_posible(self, estudiante, desafio_excluido=None):
        mejor_equipo = None
        mejor_puntaje = float('-inf')
        
        # Crear lista de desafíos disponibles
        desafios_disponibles = [desafio for desafio, equipo in self.asignaciones.items() if desafio != desafio_excluido]
        
        for desafio in desafios_disponibles:
            equipo = self.asignaciones[desafio]
            if len(equipo) < 4 and self.verificar_asignacion_valida(desafio, estudiante):
                # Calcular puntaje basado en preferencias y tamaño del equipo
                puntaje = 0
                if desafio in estudiante['Postulaciones']:
                    puntaje += (3 - estudiante['Postulaciones'].index(desafio)) * 2
                if len(equipo) == 1:  # Priorizar equipos que necesitan un miembro más
                    puntaje += 3
                if len(equipo) == 2:  # También bueno para equipos de 2
                    puntaje += 1
                # Añadir componente aleatorio al puntaje
                puntaje += random.random()
                    
                if puntaje > mejor_puntaje:
                    mejor_puntaje = puntaje
                    mejor_equipo = desafio
                    
        return mejor_equipo
    
    # Gestionar los equipos de un solo estudiante
    def resolucion_equipos_singulares(self):
        while True:
            equipos_singulares = [(desafio, equipo) for desafio, equipo in self.asignaciones.items() if len(equipo) == 1]
            
            if not equipos_singulares:
                break
            
            random.shuffle(equipos_singulares)
                
            for desafio, equipo in equipos_singulares:
                estudiante = equipo[0]
                equipo_nuevo = self.encontrar_mejor_equipo_posible(estudiante, desafio)
                
                if equipo_nuevo:
                    # Mover estudiante al nuevo equipo
                    self.asignaciones[desafio].remove(estudiante)
                    self.asignaciones[equipo_nuevo].append(estudiante)
                    self.asignacion_estudiantes[estudiante['Nombre']] = equipo_nuevo
                else:
                    # Si no se encuentra equipo, intentar traer otro estudiante
                    otros_equipos_posibles = [(des, est) for des, est in self.asignaciones.items() if des != desafio and len(est) > 2]
                    random.shuffle(otros_equipos_posibles)
                    
                    estudiante_encontrado = False
                    for otro_desafio, otro_equipo in otros_equipos_posibles:
                        compañeros_potenciales = otro_equipo.copy()
                        random.shuffle(compañeros_potenciales)
                        
                        for compañero_potencial in compañeros_potenciales:
                            if self.verificar_asignacion_valida(desafio, compañero_potencial):
                                otro_equipo.remove(compañero_potencial)
                                self.asignaciones[desafio].append(compañero_potencial)
                                self.asignacion_estudiantes[compañero_potencial['Nombre']] = desafio
                                estudiante_encontrado = True
                                break
                        if estudiante_encontrado:
                            break
                            
            # Limpiar equipos vacíos
            self.asignaciones = {des_l: est_l for des_l, est_l in self.asignaciones.items() if len(est_l) > 0}

    def asignacion_estudiantes_restantes(self):
        # Obtener y aleatorizar lista de estudiantes sin asignar
        no_asignados = [s for s in self.estudiantes if s['Nombre'] not in self.asignacion_estudiantes]
        random.shuffle(no_asignados)
        
        for estudiante in no_asignados:
            mejor_equipo = self.encontrar_mejor_equipo_posible(estudiante)
            
            if mejor_equipo:
                self.asignaciones[mejor_equipo].append(estudiante)
                self.asignacion_estudiantes[estudiante['Nombre']] = mejor_equipo
            else:
                # Aleatorizar lista de estudiantes restantes
                faltantes_restantes = [s for s in no_asignados if s != estudiante and s['Nombre'] not in self.asignacion_estudiantes]
                random.shuffle(faltantes_restantes)
                
                for faltante_restante in faltantes_restantes:
                    desafios_disponibles = [c['Titulo'] for c in self.desafios]
                    
                    for desafio_disponible in desafios_disponibles:
                        if (self.verificar_asignacion_valida(desafio_disponible, estudiante) and self.verificar_asignacion_valida(desafio_disponible, faltante_restante)):
                            self.asignaciones[desafio_disponible].extend([estudiante, faltante_restante])
                            self.asignacion_estudiantes[estudiante['Nombre']] = desafio_disponible
                            self.asignacion_estudiantes[faltante_restante['Nombre']] = desafio_disponible
                            break

    def ejecutar_distribucion(self):        
        self.estudiantes_lista_random()
        self.asignacion_inicial()
        self.resolucion_equipos_singulares()
        self.asignacion_estudiantes_restantes()
        return self.asignaciones, self.asignacion_estudiantes

    def resultados_asignacion(self):
        print("\n Resultados de asignación:")
        asignacion_valida = True
        
        desafios_a_imprimir = list(self.asignaciones.items())
        
        for desafio, equipo in desafios_a_imprimir:
            print(f"\nDesafío: {desafio}")
            print(f"Número de estudiantes: {len(equipo)}")
            
            if len(equipo) < 2:
                print("ERROR: Equipo con menos de 2 estudiantes")
                asignacion_valida = False
            elif len(equipo) > 4:
                print("ERROR: Equipo con más de 4 estudiantes")
                asignacion_valida = False
                
            cantidad_por_carrera = defaultdict(int)
            for estudiante in equipo:
                cantidad_por_carrera[estudiante['Carrera']] += 1
            
            print("Distribución por carrera:")
            for carrera, cantidad in sorted(cantidad_por_carrera.items()):
                max_permitido = self.carreras.get(carrera, 1)
                print(f"- {carrera}: {cantidad} (máximo permitido: {max_permitido})")
                if cantidad > max_permitido:
                    print(f"ERROR: Excede el máximo permitido para {carrera}")
                    asignacion_valida = False
            
            print("\nEstudiantes en el equipo:")
            miembros_equipo = equipo.copy()
            for estudiante in miembros_equipo:
                postulacion_index = (estudiante['Postulaciones'].index(desafio) + 1 if desafio in estudiante['Postulaciones'] else 0)
                preferencia = f"(Preferencia #{postulacion_index})" if postulacion_index > 0 else "(No preferido)"
                print(f"- {estudiante['Nombre']} ({estudiante['Carrera']}) {preferencia}")
        
        if asignacion_valida:
            print("\n Todas las asignaciones cumplen con las restricciones")
        else:
            print("\n Hay asignaciones que no cumplen con las restricciones")

        # Mostrar estudiantes sin asignar
        no_asignados = self.get_estudiantes_no_asignados()
        if no_asignados:
            print("\n Estudiantes sin asignar:")
            for estudiante in no_asignados:
                print(f"- {estudiante}")
                
    def get_estudiantes_no_asignados(self):
        return [s['Nombre'] for s in self.estudiantes if s['Nombre'] not in self.asignacion_estudiantes]

In [16]:
# Clase cálculo de métricas de la distribución

class MetricasDistribucion:
    def __init__(self, asignaciones, asignacion_estudiantes, estudiantes, desafios):
        self.asignaciones = asignaciones
        self.asignacion_estudiantes = asignacion_estudiantes
        self.estudiantes = estudiantes
        self.desafios = desafios

    def calcular_satisfacccion_promedio(self):
        """
        Calcula la satisfacción promedio de la distribución según (1 / preferencia_de_la_asignacion + 1) por cada estudiante
        """
        satisfaccion_total = 0
        cantidad_estudiantes = len(self.estudiantes)

        for estudiante in self.estudiantes:
            desafio_asignado = self.asignacion_estudiantes.get(estudiante['Nombre'])
            if desafio_asignado in estudiante['Postulaciones']:
                preferencia_index = estudiante['Postulaciones'].index(desafio_asignado)
                satisfaccion = 1 / (preferencia_index + 1)
            else:
                satisfaccion = 0
            satisfaccion_total += satisfaccion

        return satisfaccion_total / cantidad_estudiantes

    def cantidad_primeras_preferencias(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        estudiantes_en_prioridad_1 = 0

        for estudiante in self.estudiantes:
            if (len(estudiante['Postulaciones']) > 0 and self.asignacion_estudiantes.get(estudiante['Nombre']) == estudiante['Postulaciones'][0]):
                estudiantes_en_prioridad_1 += 1

        return estudiantes_en_prioridad_1

    def cantidad_ninguna_preferencia(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        estudiantes_fuera_de_preferencias = 0

        for estudiante in self.estudiantes:
            desafio_asignado = self.asignacion_estudiantes.get(estudiante['Nombre'])
            if desafio_asignado not in estudiante['Postulaciones']:
                estudiantes_fuera_de_preferencias += 1

        return estudiantes_fuera_de_preferencias

    def cantidad_desafios_sin_equipo_valido(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        desafios_sin_equipo = 0
        desafios_con_equipo = set(self.asignaciones.keys())

        # Contar desafíos no vacío pero no válido
        for desafio in desafios_con_equipo:
            if len(self.asignaciones[desafio]) < 2:
                desafios_sin_equipo += 1

        # Agregar desafíos con equipo vacío
        desafios_n = {desafio['Titulo'] for desafio in self.desafios}
        desafios_sin_equipo += len(desafios_n - desafios_con_equipo)

        return desafios_sin_equipo

    def calcular_equipos_STD(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        tamaño_equipos = [len(equipo) for equipo in self.asignaciones.values() if len(equipo) > 0]
        return np.std(tamaño_equipos) if tamaño_equipos else 0

    def calcular_media_de_carreras_en_equipo(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        carreras_por_equipo = []

        for equipo in self.asignaciones.values():
            if len(equipo) >= 2:  # Solo considerar equipos válidos
                carreras_unicas = len(set(estudiante['Carrera'] for estudiante in equipo))
                carreras_por_equipo.append(carreras_unicas)

        return np.mean(carreras_por_equipo) if carreras_por_equipo else 0

    def calcular_metricas(self):
        """
        Calcula todas las métricas
        """
        metricas = {
            'satisfaccion_promedio': self.calcular_satisfacccion_promedio(),
            'estudiantes_primera_prioridad': self.cantidad_primeras_preferencias(),
            'estudiantes_fuera_preferencias': self.cantidad_ninguna_preferencia(),
            'desafios_sin_equipo': self.cantidad_desafios_sin_equipo_valido(),
            'std_tamaño_equipos': self.calcular_equipos_STD(),
            'promedio_carreras_por_equipo': self.calcular_media_de_carreras_en_equipo()
        }

        return metricas

    def entregar_metricas(self):
        """
        Imprime todas las métricas
        """
        metricas = self.calcular_metricas()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metricas['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metricas['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metricas['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metricas['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metricas['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metricas['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_estudiantes = len(self.estudiantes)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metricas['estudiantes_primera_prioridad']/total_estudiantes)*100:.1f}%")
        print(f"Segunda y Tercera prioridad: {100 - (metricas['estudiantes_primera_prioridad']/total_estudiantes)*100 - (metricas['estudiantes_fuera_preferencias']/total_estudiantes)*100:.1f}%")
        print(f"Fuera de preferencias: {(metricas['estudiantes_fuera_preferencias']/total_estudiantes)*100:.1f}%")

In [18]:
# aplicar las clases con el archivo JSON

with open('body.json', 'r', encoding='utf-8') as file:
    body = json.load(file)

# Crear instancia
asignador = GenerarEquipos(body["estudiantes"], body["desafios"], body["carreras"])

# Ejecutar asignación
asignaciones, asignacion_estudiantes = asignador.ejecutar_distribucion()
# Calcular métricas
metricas = MetricasDistribucion(asignaciones, asignacion_estudiantes, body["estudiantes"],  body["desafios"])
metricas.entregar_metricas()
asignador.resultados_asignacion()

# O obtener métricas como diccionario
metricas_dict = metricas.calcular_metricas()


=== Métricas de Asignación ===
Satisfacción promedio: 0.807
Estudiantes en primera prioridad: 44
Estudiantes fuera de preferencias: 3
Desafíos sin equipo: 17
Desviación estándar tamaño equipos: 0.785
Promedio de carreras por equipo: 1.62

Porcentajes:
Primera prioridad: 68.8%
Segunda y Tercera prioridad: 26.6%
Fuera de preferencias: 4.7%

 Resultados de asignación:

Desafío: Clasificación de imágenes de mamografía usando Machine Learning
Número de estudiantes: 4
Distribución por carrera:
- Ingeniería Civil Industrial: 1 (máximo permitido: 1)
- Ingeniería Civil Telemática: 3 (máximo permitido: 3)

Estudiantes en el equipo:
- Santiago Lopez (Ingeniería Civil Telemática) (Preferencia #1)
- Marcelo Esteban Díaz Moya (Ingeniería Civil Telemática) (Preferencia #1)
- Vicente Ignacio Carreño Escobar (Ingeniería Civil Telemática) (Preferencia #1)
- Battá Tomás Tuki Cadenas (Ingeniería Civil Industrial) (Preferencia #2)

Desafío: Monitoreo de rendimiento para jugadores de rugby con tecnología L